# Chatbot Example


In this lesson, you will familiarize yourself with the chatbot example you will work on during this chapter(Chapter 4). The example includes the tool definitions and execution, as well as the chatbot code. Make sure to interact with the chatbot at the end of this notebook.


## Import Libraries


In [ ]:
!pip install openai arxiv python-dotenv -q

In [4]:
import arxiv
import json
import os
from typing import List
from dotenv import load_dotenv
from openai import OpenAI


## Tool Functions


In [5]:
PAPER_DIR = "papers"

The first tool searches for relevant arXiv papers based on a topic and stores the papers' info in a JSON file (title, authors, summary, paper url and the publication date). The JSON files are organized by topics in the `papers` directory. The tool does not download the papers.  


In [6]:
def search_papers(topic: str, max_results: int = 5) -> List[str]:
    """
    Search for papers on arXiv based on a topic and store their information.
    
    Args:
        topic: The topic to search for
        max_results: Maximum number of results to retrieve (default: 5)
        
    Returns:
        List of paper IDs found in the search
    """
    client = arxiv.Client()

    search = arxiv.Search(
        query = topic,
        max_results = max_results,
        sort_by = arxiv.SortCriterion.Relevance
    )

    papers = client.results(search)
    
    path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))
    os.makedirs(path, exist_ok=True)
    
    file_path = os.path.join(path, "papers_info.json")

    try:
        with open(file_path, "r") as json_file:
            papers_info = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError):
        papers_info = {}

    paper_ids = []
    for paper in papers:
        paper_ids.append(paper.get_short_id())
        paper_info = {
            'title': paper.title,
            'authors': [author.name for author in paper.authors],
            'summary': paper.summary,
            'pdf_url': paper.pdf_url,
            'published': str(paper.published.date())
        }
        papers_info[paper.get_short_id()] = paper_info
    
    with open(file_path, "w") as json_file:
        json.dump(papers_info, json_file, indent=2)
    
    print(f"Results are saved in: {file_path}")
    
    return paper_ids


In [7]:
search_papers("Machine Learning in Finance")

Results are saved in: papers\machine_learning_in_finance\papers_info.json


['2306.04338v1',
 '2006.16189v4',
 '2201.12150v2',
 '2302.08893v4',
 '2304.02381v2']

The second tool looks for information about a specific paper across all topic directories inside the `papers` directory.


In [8]:
def extract_info(paper_id: str) -> str:
    """
    Search for information about a specific paper across all topic directories.
    
    Args:
        paper_id: The ID of the paper to look for
        
    Returns:
        JSON string with paper information if found, error message if not found
    """
    for item in os.listdir(PAPER_DIR):
        item_path = os.path.join(PAPER_DIR, item)
        if os.path.isdir(item_path):
            file_path = os.path.join(item_path, "papers_info.json")
            if os.path.isfile(file_path):
                try:
                    with open(file_path, "r") as json_file:
                        papers_info = json.load(json_file)
                        if paper_id in papers_info:
                            return json.dumps(papers_info[paper_id], indent=2)
                except (FileNotFoundError, json.JSONDecodeError) as e:
                    print(f"Error reading {file_path}: {str(e)}")
                    continue
    
    return f"There's no saved information related to paper {paper_id}."


In [10]:
print(extract_info('2302.08893v4'))


{
  "title": "Active learning for data streams: a survey",
  "authors": [
    "Davide Cacciarelli",
    "Murat Kulahci"
  ],
  "summary": "Online active learning is a paradigm in machine learning that aims to select the most informative data points to label from a data stream. The problem of minimizing the cost associated with collecting labeled observations has gained a lot of attention in recent years, particularly in real-world applications where data is only available in an unlabeled form. Annotating each observation can be time-consuming and costly, making it difficult to obtain large amounts of labeled data. To overcome this issue, many active learning strategies have been proposed in the last decades, aiming to select the most informative observations for labeling in order to improve the performance of machine learning models. These approaches can be broadly divided into two categories: static pool-based and stream-based active learning. Pool-based active learning involves selec

## Tool Schema


Here are the schemas of each tool in **OpenAI function-calling format**.


In [11]:
# OpenAI uses a slightly different schema format compared to Anthropic.
# Tools are wrapped under a 'function' key with 'type': 'function'.
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_papers",
            "description": "Search for papers on arXiv based on a topic and store their information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "The topic to search for"
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum number of results to retrieve",
                        "default": 5
                    }
                },
                "required": ["topic"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "extract_info",
            "description": "Search for information about a specific paper across all topic directories.",
            "parameters": {
                "type": "object",
                "properties": {
                    "paper_id": {
                        "type": "string",
                        "description": "The ID of the paper to look for"
                    }
                },
                "required": ["paper_id"]
            }
        }
    }
]


## Tool Mapping


This code handles tool mapping and execution.


In [12]:
mapping_tool_function = {
    "search_papers": search_papers,
    "extract_info": extract_info
}

def execute_tool(tool_name, tool_args):
    result = mapping_tool_function[tool_name](**tool_args)

    if result is None:
        result = "The operation completed but didn't return any results."
    elif isinstance(result, list):
        result = ', '.join(result)
    elif isinstance(result, dict):
        result = json.dumps(result, indent=2)
    else:
        result = str(result)
    return result


## Chatbot Code


The chatbot handles the user's queries one by one, but it does not persist memory across the queries.


In [13]:
load_dotenv()
# OpenAI client reads OPENAI_API_KEY from environment automatically
client = OpenAI()


### Query Processing


In [14]:
def process_query(query):
    messages = [{'role': 'user', 'content': query}]

    response = client.chat.completions.create(
        model='gpt-4o',
        tools=tools,
        messages=messages
    )

    process_flag = True
    while process_flag:
        message = response.choices[0].message
        finish_reason = response.choices[0].finish_reason

        # If no tool calls, just print the text and stop
        if finish_reason == 'stop' or not message.tool_calls:
            print(message.content)
            process_flag = False

        # Handle tool calls
        elif finish_reason == 'tool_calls':
            # Add assistant message (with tool calls) to history
            messages.append(message)

            for tool_call in message.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)
                tool_call_id = tool_call.id

                print(f"Calling tool {tool_name} with args {tool_args}")
                result = execute_tool(tool_name, tool_args)

                # Append tool result to messages
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call_id,
                    "content": result
                })

            # Send updated messages back to model
            response = client.chat.completions.create(
                model='gpt-4o',
                tools=tools,
                messages=messages
            )

            # If next response is just text, print and stop
            next_message = response.choices[0].message
            if response.choices[0].finish_reason == 'stop':
                print(next_message.content)
                process_flag = False


### Chat Loop


In [16]:
def chat_loop():
    print("Type your queries or 'quit' to exit.")
    while True:
        try:
            query = input("\nQuery: ").strip()
            if query.lower() == 'quit':
                break
            process_query(query)
            print("\n")
        except Exception as e:
            print(f"\nError: {str(e)}")


Feel free to interact with the chatbot. Here's an example query: 

- Search for 2 papers on "LLM interpretability"


In [17]:
chat_loop()


Type your queries or 'quit' to exit.



Query:  find 2 papers related to RAG


Calling tool search_papers with args {'topic': 'RAG', 'max_results': 2}
Results are saved in: papers\rag\papers_info.json
Calling tool extract_info with args {'paper_id': '2511.22858v1'}
Calling tool extract_info with args {'paper_id': '2401.15391v1'}
Here are two research papers related to Retrieval-Augmented Generation (RAG):

1. **Title**: [RAG System for Supporting Japanese Litigation Procedures: Faithful Response Generation Complying with Legal Norms](https://arxiv.org/pdf/2511.22858v1)  
   **Authors**: Yuya Ishihara, Atsushi Keyaki, Hiroaki Yamada, Ryutaro Ohara, Mihoko Sumida  
   **Summary**: The paper discusses the design of a RAG-based LLM system aimed at supporting Japanese medical litigation procedures while complying with legal norms. It outlines the essential components needed for such a system, focusing on generating responses faithful to context and ensuring retrieval of appropriate external knowledge. This is particularly important in replacing expert roles in litigat


Query:  quit
